In [6]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Read training data
X_train = pd.read_csv("../data/model_inputs/X_train.csv")

y_train_feasibility = pd.read_csv(
    "../data/model_inputs/y_train_feasibility.csv"
)["feasibility_binary"]

y_train_willingness = pd.read_csv(
    "../data/model_inputs/y_train_willingness.csv"
)["willingness_binary"]

In [7]:
# 2. Define 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Logistic Regression
def run_lr_model(X, y, outcome_name):

    class_weight_options = {
        "None": None,
        "Balanced": "balanced"
    }

    all_results = {}

    for class_weight_name, class_weight_value in class_weight_options.items():

        scores = {
            "AUC": [],
            "Precision": [],
            "Recall": [],
            "Accuracy": [],
            "F1": []
        }

        for train_idx, val_idx in cv.split(X, y):

            X_train_fold = X.iloc[train_idx]
            y_train_fold = y.iloc[train_idx]

            X_val_fold = X.iloc[val_idx]
            y_val_fold = y.iloc[val_idx]

            lr_model = Pipeline([
                ("scaler", StandardScaler()),
                ("logistic_regression", LogisticRegression(
                    max_iter=5000,
                    random_state=42,
                    class_weight=class_weight_value
                ))
            ])

            lr_model.fit(X_train_fold, y_train_fold)

            # Default threshold = 0.5
            y_val_pred = lr_model.predict(X_val_fold)
            y_val_prob = lr_model.predict_proba(X_val_fold)[:, 1]

            scores["AUC"].append(
                roc_auc_score(y_val_fold, y_val_prob)
            )

            scores["Precision"].append(
                precision_score(
                    y_val_fold,
                    y_val_pred,
                    zero_division=0
                )
            )

            scores["Recall"].append(
                recall_score(
                    y_val_fold,
                    y_val_pred,
                    zero_division=0
                )
            )

            scores["Accuracy"].append(
                accuracy_score(
                    y_val_fold,
                    y_val_pred
                )
            )

            scores["F1"].append(
                f1_score(
                    y_val_fold,
                    y_val_pred,
                    zero_division=0
                )
            )
            
        all_results[class_weight_name] = {
            "class_weight": class_weight_value,
            "scores": scores,
            "mean_auc": np.mean(scores["AUC"])
        }

    # Select class weight using mean CV AUC
    best_class_weight_name = max(
        all_results,
        key=lambda option: all_results[option]["mean_auc"]
    )

    selected_result = all_results[best_class_weight_name]

    best_class_weight = selected_result["class_weight"]
    best_scores = selected_result["scores"]

    print(
        f"\nSelected class weight for {outcome_name}: "
        f"{best_class_weight_name}"
    )

    metric_order = [
        "AUC",
        "Precision",
        "Recall",
        "Accuracy",
        "F1"
    ]

    cv_summary = pd.DataFrame({
        "Metric": metric_order,
        "CV Score (SE)": [
            (
                f"{np.mean(best_scores[metric]):.3f} "
                f"({np.std(best_scores[metric], ddof=1) / np.sqrt(5):.3f})"
            )
            for metric in metric_order
        ]
    })

    print(
        f"\n===== {outcome_name} Logistic Regression: 5-fold CV ====="
    )

    print(cv_summary.to_string(index=False))

    return {
        "cv_summary": cv_summary,
        "best_class_weight": best_class_weight,
        "best_class_weight_name": best_class_weight_name
    }

In [8]:
# 4. Feasibility Logistic Regression
feasibility_lr_results = run_lr_model(
    X=X_train,
    y=y_train_feasibility,
    outcome_name="Feasibility"
)

# 5. Willingness Logistic Regression
willingness_lr_results = run_lr_model(
    X=X_train,
    y=y_train_willingness,
    outcome_name="Willingness"
)


Selected class weight for Feasibility: Balanced

===== Feasibility Logistic Regression: 5-fold CV =====
   Metric CV Score (SE)
      AUC 0.622 (0.007)
Precision 0.819 (0.005)
   Recall 0.624 (0.004)
 Accuracy 0.604 (0.005)
       F1 0.708 (0.004)

Selected class weight for Willingness: Balanced

===== Willingness Logistic Regression: 5-fold CV =====
   Metric CV Score (SE)
      AUC 0.704 (0.009)
Precision 0.912 (0.003)
   Recall 0.689 (0.006)
 Accuracy 0.678 (0.005)
       F1 0.785 (0.004)


In [9]:
# 6. Read independent test data
X_test = pd.read_csv("../data/model_inputs/X_test.csv")

y_test_feasibility = pd.read_csv(
    "../data/model_inputs/y_test_feasibility.csv"
)["feasibility_binary"]

y_test_willingness = pd.read_csv(
    "../data/model_inputs/y_test_willingness.csv"
)["willingness_binary"]


# 7. Independent test-set evaluation
def evaluate_lr_test(
    X_train,
    y_train,
    X_test,
    y_test,
    outcome_name,
    best_class_weight
):

    final_lr = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            max_iter=5000,
            random_state=42,
            class_weight=best_class_weight
        ))
    ])

    final_lr.fit(X_train, y_train)

    # Default threshold = 0.5
    y_test_pred = final_lr.predict(X_test)
    y_test_prob = final_lr.predict_proba(X_test)[:, 1]

    test_summary = pd.DataFrame({
        "Metric": [
            "AUC",
            "Precision",
            "Recall",
            "Accuracy",
            "F1"
        ],
        "Test Score": [
            roc_auc_score(y_test, y_test_prob),
            precision_score(
                y_test,
                y_test_pred,
                zero_division=0
            ),
            recall_score(
                y_test,
                y_test_pred,
                zero_division=0
            ),
            accuracy_score(y_test, y_test_pred),
            f1_score(
                y_test,
                y_test_pred,
                zero_division=0
            )
        ]
    })

    test_summary["Test Score"] = test_summary[
        "Test Score"
    ].round(3)

    print(
        f"\n===== {outcome_name}: Independent test performance ====="
    )
    print(test_summary.to_string(index=False))

    return {
        "model": final_lr,
        "test_summary": test_summary
    }

In [10]:
# 8. Feasibility LR test evaluation
lr_feasibility_test_results = evaluate_lr_test(
    X_train=X_train,
    y_train=y_train_feasibility,
    X_test=X_test,
    y_test=y_test_feasibility,
    outcome_name="Feasibility Logistic Regression",
    best_class_weight=feasibility_lr_results[
        "best_class_weight"
    ]
)


# 9. Willingness LR test evaluation
lr_willingness_test_results = evaluate_lr_test(
    X_train=X_train,
    y_train=y_train_willingness,
    X_test=X_test,
    y_test=y_test_willingness,
    outcome_name="Willingness Logistic Regression",
    best_class_weight=willingness_lr_results[
        "best_class_weight"
    ]
)


===== Feasibility Logistic Regression: Independent test performance =====
   Metric  Test Score
      AUC       0.615
Precision       0.824
   Recall       0.637
 Accuracy       0.616
       F1       0.719

===== Willingness Logistic Regression: Independent test performance =====
   Metric  Test Score
      AUC       0.691
Precision       0.905
   Recall       0.688
 Accuracy       0.672
       F1       0.782
